# Sudoku qualitative study (8 ablations × 3 seeds = 22 alive checkpoints)

This notebook is the post-trajectory-dump analysis surface for the Sudoku qualitative
study described in `SUDOKU_ANALYSIS_COMMANDS.md`. It assumes you have already produced
the consolidated parquet at `outputs/sudoku_analysis/sudoku_analysis.parquet` via
`submit_sudoku_qualitative_dumps.sh` + `python -m relay.n_way_pair_trajectories`.

Sections:

1. **Bootstrap & load parquet** — add repo root to `sys.path`, load the table, derive `n_clues` / `rating_quartile`.
2. **Cohort selection** — drop the two diverged seed-tuples (`mlm_uniform_tied_seed3`, `relay_sg_tied_seed1`; legacy fallback drops the entire tied-StopGrad cell when no `seed` column is present), optional brute-force exclusion sub-study, optional stratified equal-N-per-tier subsample. All downstream figures consume the resulting `df_eval`.
3. **F1 — Pareto frontier**: exact match vs mean rollout steps (NFE).
4. **F2 — Difficulty-conditioned gain (Relay vs baselines)**: Δ exact match by `n_clues` quartile (primary), `rating` quartile (secondary), and `hardest_strategy` (tertiary).
5. **F3 — Solver alignment**: Spearman ρ of fill order, prefix Jaccard at *k*.
6. **F4 — Mechanism: legality preservation + late-step purity**: trajectory-level violation rates, final-board legality rate, late-step purity, plus the original step-1 primitive composition.
7. **F5 — Qualitative gallery**: clue grid + solver fill-time + per-model fill-time + final answer.
8. **Matched-NFE comparison**: interpolate the τ sweep to a common NFE budget. With per-(model, tying) NFE-range diagnostic so you can see when interpolation is actually bracketed.

Most of follow-ups (1)-(4) from the prior interpretation are implemented in the cohort cell, the F2 rewrite, the F4 rewrite, and the matched-NFE diagnostic. (3) — wider τ sweep — is a re-dump step (see `SUDOKU_ANALYSIS_COMMANDS.md`), but the notebook auto-adapts to whatever τs are present.

In [ ]:
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "relay" / "sudoku_analysis.py").is_file():
            return p
    raise RuntimeError("Run from inside the double-backprop repo")

_root = _repo_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from relay import sudoku_analysis as sa
from relay.viz_puzzle_state import (
    overlay_sudoku_digits_on_heatmap_ax,
)

PARQUET = _root / "outputs" / "sudoku_analysis" / "sudoku_analysis.parquet"
FIGS = _root / "outputs" / "sudoku_analysis" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
print("parquet:", PARQUET, PARQUET.exists())
print("figs:   ", FIGS)

In [ ]:
df = pd.read_parquet(PARQUET)

# Seed support (added by the seeded sweep; older 8-run parquets won't have it).
if "seed" not in df.columns:
    df["seed"] = pd.NA
df["seed"] = df["seed"].astype("Int64")  # nullable int

print(
    "rows:", len(df),
    "| unique runs:", df["run_id"].nunique(),
    "| unique tau:", sorted(df["tau"].unique()),
    "| unique seeds:", sorted([int(s) for s in df["seed"].dropna().unique().tolist()]),
    "| unique puzzles:", df["puzzle_hash"].nunique(),
)

OBJECTIVE_ORDER = [
    "mlm_uniform",
    "rollout",
    "relay_sg",
    "relay",
]
OBJECTIVE_LABEL = {
    "mlm_uniform": "Uniform MLM",
    "rollout": "No-Relay (Rollout-buffer-only)",
    "relay_sg": "Relay (StopGrad)",
    "relay": "Relay (BPTT)",
}
OBJECTIVE_COLOR = {
    "mlm_uniform": "#888888",
    "rollout": "#1f77b4",
    "relay_sg": "#ff7f0e",
    "relay": "#d62728",
}
STRATEGY_ORDER = ["Easy", "Medium", "Advanced", "Master", "BruteForce"]
df["hardest_strategy"] = pd.Categorical(
    df["hardest_strategy"], categories=STRATEGY_ORDER, ordered=True
)
df["objective"] = pd.Categorical(df["objective"], categories=OBJECTIVE_ORDER, ordered=True)

# Derived difficulty axes (used in F2). `n_clues` is already populated per row in the parquet;
# `rating` is the timvink-solver difficulty score (higher = harder). We bucket both into quartiles
# so we can use them as ordered categorical primary axes when strategy tiers are too sparse.
def _quartile_label(s: pd.Series, name: str) -> pd.Series:
    # Use unique edges so we don't crash on degenerate distributions; label as "[lo, hi]".
    try:
        q = pd.qcut(s, q=4, duplicates="drop")
    except ValueError:
        return pd.Series([f"all_{name}"] * len(s), index=s.index)
    return q.astype(str)

df["n_clues_quartile"] = _quartile_label(df["n_clues"].astype(float), "n_clues")
df["rating_quartile"] = _quartile_label(df["rating"].astype(float), "rating")

# Puzzle-level tier counts so the cohort cell knows how many puzzles are available per tier.
puzzle_tiers = (
    df.drop_duplicates("puzzle_hash")[["puzzle_hash", "hardest_strategy"]]
    .groupby("hardest_strategy", observed=True)
    .size()
    .reindex(STRATEGY_ORDER, fill_value=0)
)
print("puzzle-level tier counts:", puzzle_tiers.to_dict())
df.head()

## Cohort selection

Builds `df_eval` — the working DataFrame consumed by every figure below. Three knobs:

- `DROP_DIVERGED_TIED_STOPGRAD` (**True** by default) — drops `(objective="relay_sg", embed_tying="tied")` rows. Once you've re-run the seeded sweep this can be flipped off (the diverged run will be replaced with non-diverged seeds).
- `DROP_BRUTE_FORCE` (False by default) — sub-study toggle for "filter out brute-force puzzles". Flip on to compare only puzzles that timvink-solver labelled with a human-style strategy (Easy/Medium/Advanced/Master). Useful because the model is not doing recursive backtracking.
- `STRATIFY_BY_TIER` (False by default; flip on once the larger eval slice has Advanced/Master coverage) — equal-N stratified subsample per `hardest_strategy` tier. Lets you compare equal numbers of Easy / Medium / Advanced / Master / BruteForce puzzles. With the default 200-puzzle slice (158 BF, 36 Easy, 6 Medium, 0 Adv/Master) this caps you at min populated tier (≈6) and is mostly informative once a bigger slice is dumped.

The `COHORT_LABEL` variable threads through every figure title so the toggles you used are always visible in the saved PNG.

In [ ]:
DROP_DIVERGED_SEEDS = True
DROP_BRUTE_FORCE = False
STRATIFY_BY_TIER = False
STRATIFY_N_PER_TIER: int | str = "min"  # int, or "min" = cap at min populated-tier count.
STRATIFY_RANDOM_SEED = 0

# (objective, embed_tying, seed) tuples that diverged at training time
# (val/prediction/legal_rate == 0 in W&B). With the seeded sweep we drop only
# the bad seed instead of the entire (objective, tying) cell -- the
# corresponding W&B project is `<entity>/BPTT-sudoku` (entity withheld for
# double-blind review).
DIVERGED_SEED_TUPLES: list[tuple[str, str, int]] = [
    ("mlm_uniform", "tied", 3),
    ("relay_sg", "tied", 1),
]
# Legacy fallback when the parquet has no `seed` column (older 8-run dumps):
# drop the entire (relay_sg, tied) cell, which corresponds to
# the original `v6nzrrye` divergence at the Hydra-default seed=1.
LEGACY_DROP_OBJECTIVE_TYING: tuple[str, str] = ("relay_sg", "tied")

df_eval = df.copy()
cohort_bits: list[str] = []

if DROP_DIVERGED_SEEDS:
    has_seed = df_eval["seed"].notna().any()
    if has_seed:
        n_total = 0
        for obj, ty, sd in DIVERGED_SEED_TUPLES:
            mask = (
                (df_eval["objective"] == obj)
                & (df_eval["embed_tying"] == ty)
                & (df_eval["seed"] == sd)
            )
            n_dropped = int(mask.sum())
            n_total += n_dropped
            df_eval = df_eval[~mask].copy()
        cohort_bits.append(
            f"-diverged-seeds ({n_total} rows; tuples={DIVERGED_SEED_TUPLES})"
        )
    else:
        obj, ty = LEGACY_DROP_OBJECTIVE_TYING
        mask = (df_eval["objective"] == obj) & (df_eval["embed_tying"] == ty)
        n_dropped = int(mask.sum())
        df_eval = df_eval[~mask].copy()
        cohort_bits.append(f"-{ty}-{obj} legacy fallback ({n_dropped} rows)")

if DROP_BRUTE_FORCE:
    mask = df_eval["hardest_strategy"] == "BruteForce"
    n_puzzles_dropped = df_eval.loc[mask, "puzzle_hash"].nunique()
    df_eval = df_eval[~mask].copy()
    cohort_bits.append(f"-BruteForce ({n_puzzles_dropped} puzzles)")

if STRATIFY_BY_TIER:
    rng = np.random.default_rng(STRATIFY_RANDOM_SEED)
    puzzle_tiers = (
        df_eval.drop_duplicates("puzzle_hash")[["puzzle_hash", "hardest_strategy"]]
        .dropna(subset=["hardest_strategy"])
    )
    tier_counts = (
        puzzle_tiers.groupby("hardest_strategy", observed=True).size()
        .reindex(STRATEGY_ORDER, fill_value=0)
    )
    populated = tier_counts[tier_counts > 0]
    if isinstance(STRATIFY_N_PER_TIER, str) and STRATIFY_N_PER_TIER == "min":
        n_per_tier = int(populated.min())
    else:
        n_per_tier = int(min(STRATIFY_N_PER_TIER, populated.min()))
    keep_hashes: list[str] = []
    for tier in populated.index:
        cand = puzzle_tiers[puzzle_tiers["hardest_strategy"] == tier]
        idx = rng.choice(cand["puzzle_hash"].values, size=n_per_tier, replace=False)
        keep_hashes.extend(idx.tolist())
    df_eval = df_eval[df_eval["puzzle_hash"].isin(keep_hashes)].copy()
    cohort_bits.append(
        f"stratified n_per_tier={n_per_tier} across {len(populated)} tiers ({populated.index.tolist()})"
    )

# Cohort label that we thread into every figure title.
if cohort_bits:
    COHORT_LABEL = "cohort: " + ", ".join(cohort_bits)
else:
    COHORT_LABEL = "cohort: full"

print(COHORT_LABEL)
seed_summary = (
    sorted([int(s) for s in df_eval["seed"].dropna().unique().tolist()])
    if df_eval["seed"].notna().any()
    else "[]"
)
print(
    f"df_eval rows: {len(df_eval)}",
    f"| unique runs: {df_eval['run_id'].nunique()}",
    f"| unique seeds: {seed_summary}",
    f"| unique puzzles: {df_eval['puzzle_hash'].nunique()}",
    f"| unique tau: {sorted(df_eval['tau'].unique())}",
)
print("\nper-tier puzzle counts in df_eval:")
print(
    df_eval.drop_duplicates("puzzle_hash")["hardest_strategy"]
    .value_counts(dropna=False)
    .reindex(STRATEGY_ORDER, fill_value=0)
)
if df_eval["seed"].notna().any():
    print("\n(objective, embed_tying, seed) cells in df_eval:")
    print(
        df_eval.drop_duplicates(["run_id"])
        .groupby(["objective", "embed_tying", "seed"], observed=True)
        .size()
        .rename("n_runs_x_tau")
        .reset_index()
    )

## F1 — Pareto frontier

Exact match (and optionally token accuracy) vs **mean rollout steps**, one curve per
(`objective` x `embed_tying`) pair, threshold marker size scaled by τ. Reproduces the
screenshot from the paper writeup directly from the same evaluation slice the rest
of the analyses use — so changes in the curves track changes in the dump set.

In [ ]:
agg = (
    df_eval.groupby(["objective", "embed_tying", "tau"], observed=True)
    .agg(exact_match=("exact_match", "mean"),
         rollout_steps=("rollout_steps", "mean"),
         n=("exact_match", "size"))
    .reset_index()
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, tying in zip(axes, ["untied", "tied"]):
    sub = agg[agg["embed_tying"] == tying]
    for obj in OBJECTIVE_ORDER:
        s = sub[sub["objective"] == obj].sort_values("tau")
        if s.empty:
            continue
        ax.plot(s["rollout_steps"], s["exact_match"], "o-",
                color=OBJECTIVE_COLOR[obj], label=OBJECTIVE_LABEL[obj])
        for _, row in s.iterrows():
            ax.annotate(f"τ={row['tau']:.2f}",
                        (row["rollout_steps"], row["exact_match"]),
                        textcoords="offset points", xytext=(4, 4), fontsize=7)
    ax.set_xlabel("Mean rollout steps (NFE)")
    ax.set_title(f"embed_tying = {tying}")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Exact match")
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle(f"F1 — Sudoku Pareto frontier (exact match vs NFE) — {COHORT_LABEL}", y=1.02)
fig.tight_layout()
fig.savefig(FIGS / "F1_pareto.png", dpi=180, bbox_inches="tight")
fig.savefig(FIGS / "F1_pareto.pdf", bbox_inches="tight")
plt.show()
agg

## F2 — Difficulty-conditioned gain (Relay − baseline)

Δ exact match against three difficulty axes:

1. **`n_clues` quartile** (primary): clues are a model-agnostic difficulty signal. With ~25 clue Sudoku Extreme puzzles, lower-clue puzzles are systematically harder.
2. **`rating` quartile** (secondary): timvink-solver's tier-weighted rating — a more refined difficulty signal than `hardest_strategy`.
3. **`hardest_strategy`** (tertiary): kept for completeness, but with the standard test slice this axis is dominated by `BruteForce` (heuristic fallback) and the `Advanced`/`Master` tiers are typically empty. Note in the figure title gives the populated-tier counts.

In [ ]:
TARGET_TAU = 0.15
F2_TYING = "untied"  # Tied panel is unstable when StopGrad-tied has diverged.
BASELINES = ["mlm_uniform", "rollout", "relay_sg"]
BASELINE_COLOR = {b: OBJECTIVE_COLOR[b] for b in BASELINES}

def per_puzzle_em(obj: str, tying: str, tau: float = TARGET_TAU) -> pd.Series:
    sub = df_eval[
        (df_eval["objective"] == obj) & (df_eval["embed_tying"] == tying) & (df_eval["tau"] == tau)
    ]
    return sub.set_index("puzzle_hash")["exact_match"].astype(float)

def difficulty_meta_for_lear(tying: str = F2_TYING, tau: float = TARGET_TAU) -> pd.DataFrame:
    """One row per puzzle covered by Relay at the given (tying, tau)."""
    lear_rows = df_eval[
        (df_eval["objective"] == "relay")
        & (df_eval["embed_tying"] == tying)
        & (df_eval["tau"] == tau)
    ]
    return lear_rows.set_index("puzzle_hash")[
        ["n_clues", "n_clues_quartile", "rating", "rating_quartile", "hardest_strategy"]
    ]

def deltas_for_axis(tying: str = F2_TYING, tau: float = TARGET_TAU) -> pd.DataFrame:
    rows = []
    lear = per_puzzle_em("relay", tying, tau)
    if lear.empty:
        return pd.DataFrame()
    meta = difficulty_meta_for_lear(tying, tau)
    for baseline in BASELINES:
        base = per_puzzle_em(baseline, tying, tau)
        common = lear.index.intersection(base.index)
        if common.empty:
            continue
        diff = (lear.loc[common] - base.loc[common]).rename("delta")
        m = meta.loc[common]
        merged = pd.concat([diff, m], axis=1).reset_index()
        merged["baseline"] = baseline
        rows.append(merged)
    return pd.concat(rows, axis=0, ignore_index=True) if rows else pd.DataFrame()

delta_df = deltas_for_axis()

def _bar_by_axis(ax, sub: pd.DataFrame, axis_col: str, axis_order=None, title=""):
    means = (
        sub.groupby([axis_col, "baseline"], observed=True)["delta"]
        .mean()
        .unstack("baseline")
    )
    if axis_order is not None:
        means = means.reindex(axis_order)
    keep = means.dropna(how="all")
    if keep.empty:
        ax.text(0.5, 0.5, "no data", ha="center", va="center",
                transform=ax.transAxes, color="#888")
        ax.set_xticks([]); ax.set_yticks([])
        return
    cols = [b for b in BASELINES if b in keep.columns]
    keep[cols].plot(kind="bar", ax=ax, width=0.8,
                    color=[BASELINE_COLOR[b] for b in cols],
                    legend=False)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(axis_col)
    ax.tick_params(axis="x", labelrotation=20)
    # Per-bin n
    counts = sub.groupby([axis_col], observed=True)["puzzle_hash"].nunique()
    if axis_order is not None:
        counts = counts.reindex(axis_order)
    counts = counts.reindex(keep.index)
    for i, (idx, n) in enumerate(counts.items()):
        if pd.notna(n):
            ax.text(i, ax.get_ylim()[1] * 0.92, f"n={int(n)}",
                    ha="center", fontsize=7, color="#444")

if not delta_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
    n_total_puzzles = delta_df["puzzle_hash"].nunique()

    n_clues_order = (
        delta_df.dropna(subset=["n_clues_quartile"])
        .sort_values("n_clues")["n_clues_quartile"].drop_duplicates().tolist()
    )
    rating_order = (
        delta_df.dropna(subset=["rating_quartile"])
        .sort_values("rating")["rating_quartile"].drop_duplicates().tolist()
    )

    _bar_by_axis(axes[0], delta_df, "n_clues_quartile",
                 axis_order=n_clues_order,
                 title="a) by n_clues quartile (primary)")
    _bar_by_axis(axes[1], delta_df, "rating_quartile",
                 axis_order=rating_order,
                 title="b) by rating quartile (secondary)")
    _bar_by_axis(axes[2], delta_df, "hardest_strategy",
                 axis_order=STRATEGY_ORDER,
                 title="c) by hardest_strategy (tertiary)")
    axes[0].set_ylabel("Δ exact match (Relay − baseline)")
    handles = [
        plt.Line2D([0], [0], color=BASELINE_COLOR[b], lw=8,
                   label=OBJECTIVE_LABEL[b])
        for b in BASELINES
    ]
    axes[-1].legend(handles=handles, fontsize=8, loc="lower right")
    fig.suptitle(
        f"F2 — Difficulty-conditioned Relay gain  ({F2_TYING}, τ={TARGET_TAU:.2f}, "
        f"n={n_total_puzzles} puzzles, {COHORT_LABEL})",
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(FIGS / "F2_difficulty_gain.png", dpi=180, bbox_inches="tight")
    plt.show()

    print("\nRelay EM by n_clues quartile (untied, τ=0.15):")
    em_by_clues = (
        df_eval[
            (df_eval["objective"] == "relay")
            & (df_eval["embed_tying"] == F2_TYING)
            & (df_eval["tau"] == TARGET_TAU)
        ]
        .groupby("n_clues_quartile", observed=True)
        .agg(em=("exact_match", "mean"), n=("exact_match", "size"))
    )
    print(em_by_clues)
delta_df.head() if not delta_df.empty else "(empty)"

## F3 — Solver alignment

Two process-level alignment numbers (already pre-computed per row in the parquet):

- `fill_order_spearman`: rank correlation of model fill-step grid vs solver fill-step grid (excludes clue cells).
- `prefix_jaccard_at_{5,10,20}`: cells filled in the first k model steps vs first k solver steps.

In [ ]:
tau15 = df_eval[df_eval["tau"] == TARGET_TAU].copy()
if not tau15.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Panel (a): Spearman ρ by objective (averaged over puzzles), split by tying.
    spearman = (
        tau15.groupby(["objective", "embed_tying"], observed=True)["fill_order_spearman"]
        .mean()
        .unstack("embed_tying")
    )
    objs_present = [o for o in OBJECTIVE_ORDER if o in spearman.index]
    spearman = spearman.loc[objs_present]
    spearman.plot(kind="bar", ax=axes[0], width=0.7,
                  color=["#4c4c4c", "#bbbbbb"])
    axes[0].set_ylabel("Spearman ρ (fill order vs solver)")
    axes[0].set_xlabel("")
    axes[0].set_title("a) Fill-order rank correlation")
    axes[0].set_xticklabels([OBJECTIVE_LABEL[o] for o in objs_present],
                            rotation=20, ha="right")
    axes[0].legend(title="embed_tying", fontsize=8)
    axes[0].axhline(0, color="black", linewidth=0.5)

    # Panel (b): prefix Jaccard at increasing k.
    long = tau15.melt(
        id_vars=["objective", "hardest_strategy", "embed_tying"],
        value_vars=["prefix_jaccard_at_5", "prefix_jaccard_at_10", "prefix_jaccard_at_20"],
        var_name="k",
        value_name="jaccard",
    )
    long["k"] = long["k"].str.replace("prefix_jaccard_at_", "").astype(int)
    pivot = (
        long.groupby(["k", "objective"], observed=True)["jaccard"]
        .mean()
        .unstack("objective")
    )
    objs_present_p = [o for o in OBJECTIVE_ORDER if o in pivot.columns]
    pivot[objs_present_p].plot(
        ax=axes[1], marker="o", color=[OBJECTIVE_COLOR[o] for o in objs_present_p]
    )
    axes[1].set_ylabel("Jaccard(model first-k, solver first-k)")
    axes[1].set_xlabel("k (steps)")
    axes[1].set_title("b) Prefix overlap with solver")
    axes[1].legend([OBJECTIVE_LABEL[o] for o in objs_present_p], fontsize=7)
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(
        f"F3 — Solver alignment (τ={TARGET_TAU:.2f}, {COHORT_LABEL})",
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(FIGS / "F3_solver_alignment.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("No rows at tau =", TARGET_TAU)

## F4 — Mechanism: legality preservation + late-step purity

Reframed from the original "step-1 primitives" cut (which showed near-identical
choices across models) to focus on what *actually* differs between the four
objectives over the full rollout. Four panels:

- **(a)** *Trajectory constraint violations* — mean violations summed across all
  rollout steps, and mean final-board violations. Relay-trained models maintain
  legality far better mid-trajectory than non-relay.
- **(b)** *Final-board legality rate* — fraction of puzzles whose final 81-cell
  prediction has zero row/column/box duplicates. Mirrors the new
  `legal_rate` validation metric introduced in `SUDOKU_COMMANDS.md`.
- **(c)** *Per-step purity curve* — fraction of newly-committed cells that are correct
  at each rollout step. Late-step purity is where relay's gain shows up.
- **(d)** *Step-1 primitive composition* — kept for reference; this is where the
  original "smarter first cells" claim came from but is largely flat across models.

In [ ]:
if not tau15.empty:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    objs_present = [o for o in OBJECTIVE_ORDER if (tau15["objective"] == o).any()]
    obj_colors = [OBJECTIVE_COLOR[o] for o in objs_present]
    obj_labels = [OBJECTIVE_LABEL[o] for o in objs_present]

    # Panel (a): trajectory + final violations.
    legality = (
        tau15.groupby("objective", observed=True)[["violations_total_steps", "final_violations"]]
        .mean()
        .reindex(objs_present)
    )
    legality.plot(kind="bar", ax=axes[0, 0], width=0.8,
                  color=["#2ca02c", "#d62728"])
    axes[0, 0].set_xticklabels(obj_labels, rotation=20, ha="right")
    axes[0, 0].set_ylabel("mean violations / puzzle")
    axes[0, 0].set_title("a) Constraint violations across the rollout")
    axes[0, 0].legend(["sum over all steps", "final board"], fontsize=8)
    for i, (idx, row) in enumerate(legality.iterrows()):
        axes[0, 0].text(i - 0.18, row["violations_total_steps"] + 0.3,
                        f"{row['violations_total_steps']:.1f}",
                        ha="center", fontsize=7)
        axes[0, 0].text(i + 0.18, row["final_violations"] + 0.3,
                        f"{row['final_violations']:.1f}",
                        ha="center", fontsize=7)

    # Panel (b): final-board legality rate.
    legal_rate = (
        tau15.assign(_legal=(tau15["final_violations"] == 0).astype(float))
        .groupby("objective", observed=True)["_legal"]
        .mean()
        .reindex(objs_present)
    )
    legal_rate.plot(kind="bar", ax=axes[0, 1], color=obj_colors, width=0.7)
    axes[0, 1].set_xticklabels(obj_labels, rotation=20, ha="right")
    axes[0, 1].set_ylabel("frac. final boards with 0 violations")
    axes[0, 1].set_title("b) Final-board legality rate")
    axes[0, 1].set_ylim(0, 1.05)
    for i, v in enumerate(legal_rate.values):
        axes[0, 1].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)

    # Panel (c): per-step purity (extended view + late-step focus).
    purity_lookup = []
    for _, row in tau15.iterrows():
        purity = row["purity_per_step"]
        if purity is None:
            continue
        for t, p in enumerate(purity):
            if p is None or pd.isna(p):
                continue
            purity_lookup.append({
                "objective": row["objective"],
                "step": t + 1,
                "purity": float(p),
            })
    purity_df = pd.DataFrame(purity_lookup)
    if not purity_df.empty:
        purity_pivot = (
            purity_df.groupby(["step", "objective"], observed=True)["purity"]
            .mean()
            .unstack("objective")
        )
        steps_to_show = purity_pivot.index[:40]
        objs_pp = [o for o in objs_present if o in purity_pivot.columns]
        purity_pivot.loc[steps_to_show, objs_pp].plot(
            ax=axes[1, 0], marker="o",
            color=[OBJECTIVE_COLOR[o] for o in objs_pp],
        )
        axes[1, 0].set_ylabel("per-step purity (correct / new)")
        axes[1, 0].set_xlabel("decoding step")
        axes[1, 0].set_title("c) Per-step purity (steps 1–40)")
        axes[1, 0].legend([OBJECTIVE_LABEL[o] for o in objs_pp], fontsize=7,
                          loc="lower left")
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].set_ylim(0, 1.05)

        # Inset: late-step purity = mean over the last 5 steps each model actually decoded for.
        late_purity = (
            purity_df.assign(
                model_max_step=lambda x: x.groupby("objective", observed=True)["step"].transform("max")
            )
            .pipe(lambda x: x[x["step"] >= x["model_max_step"] - 4])
            .groupby("objective", observed=True)["purity"]
            .mean()
            .reindex(objs_present)
        )
        # Use ax inset for the late-step bar
        ax_inset = axes[1, 0].inset_axes([0.55, 0.20, 0.42, 0.36])
        late_purity.plot(kind="bar", ax=ax_inset, color=obj_colors, width=0.7)
        ax_inset.set_xticklabels([OBJECTIVE_LABEL[o].split(" ")[0] for o in late_purity.index],
                                 rotation=20, ha="right", fontsize=6)
        ax_inset.set_title("late-step purity (last 5 steps)", fontsize=7)
        ax_inset.set_ylabel("", fontsize=6)
        ax_inset.tick_params(labelsize=6)
        ax_inset.set_ylim(0, 1.0)

    # Panel (d): step-1 primitive composition (de-emphasized).
    primitives = (
        tau15.groupby("objective", observed=True)[
            [
                "first_step_naked_single_rate",
                "first_step_hidden_single_rate",
                "first_step_advanced_rate",
                "first_step_illegal_rate",
            ]
        ]
        .mean()
        .reindex(objs_present)
    )
    primitives.plot(kind="bar", ax=axes[1, 1], width=0.85,
                    color=["#1f77b4", "#2ca02c", "#9467bd", "#d62728"])
    axes[1, 1].set_xticklabels(obj_labels, rotation=20, ha="right")
    axes[1, 1].set_ylabel("frac. of step-1 cells")
    axes[1, 1].set_title("d) Step-1 primitive composition (reference; ~flat across models)")
    axes[1, 1].legend(["naked single", "hidden single", "advanced", "illegal"], fontsize=7)

    fig.suptitle(
        f"F4 — Mechanism: legality + late-step purity  (τ={TARGET_TAU:.2f}, {COHORT_LABEL})",
        y=1.005,
    )
    fig.tight_layout()
    fig.savefig(FIGS / "F4_mechanism.png", dpi=180, bbox_inches="tight")
    plt.show()
    print("Mean violations:")
    print(legality.round(2))
    print("\nFinal-board legality rate:")
    print(legal_rate.round(3))

## F5 — Qualitative gallery

For 4–6 representative puzzles (one per strategy tier), show, side-by-side,
the clue grid, the solver fill-time map, and per-model fill-time maps with
overlaid digits via `overlay_sudoku_digits_on_heatmap_ax`. Cells that the
model gets wrong at the end are highlighted in red.

In [ ]:
def grid_from_flat(flat, default=0):
    arr = np.asarray(flat, dtype=np.float32)
    if arr.size != 81:
        return None
    return arr.reshape(9, 9)

def digit_grid_from_flat(flat):
    arr = np.asarray(flat, dtype=np.int8).reshape(9, 9)
    return arr

def to_label_grid(int_grid):
    return np.array(
        [["" if int(v) == 0 else str(int(v)) for v in row] for row in int_grid],
        dtype=object,
    )

def pick_one_per_tier(target_tau=TARGET_TAU, max_per_tier=1):
    sub = df_eval[(df_eval["tau"] == target_tau) & (df_eval["objective"] == "relay")]
    chosen = []
    for tier in STRATEGY_ORDER:
        cand = sub[sub["hardest_strategy"] == tier]
        if cand.empty:
            continue
        chosen.append(cand.sample(n=min(max_per_tier, len(cand)), random_state=0))
    if not chosen:
        return pd.DataFrame()
    return pd.concat(chosen, axis=0).reset_index(drop=True)

def render_gallery(seed_rows, target_tau=TARGET_TAU):
    if seed_rows.empty:
        print("(no seed puzzles found)")
        return
    cols = ["clues", "solver"] + [OBJECTIVE_LABEL[o] for o in OBJECTIVE_ORDER]
    n_rows = len(seed_rows)
    fig, axes = plt.subplots(
        n_rows, len(cols),
        figsize=(2.4 * len(cols), 2.4 * n_rows),
        squeeze=False,
    )
    cmap = plt.get_cmap("viridis")
    for r, (_, seed) in enumerate(seed_rows.iterrows()):
        ph = seed["puzzle_hash"]
        clue_grid = digit_grid_from_flat(seed["clue_grid_flat"])
        answer = digit_grid_from_flat(seed["answer_grid_flat"])
        solver_fs = grid_from_flat(seed.get("solver_fill_step_flat"))
        # Column 0: clue grid.
        ax = axes[r, 0]
        ax.imshow(np.zeros((9, 9)), cmap=ListedColormap(["#ffffff"]))
        overlay_sudoku_digits_on_heatmap_ax(ax, to_label_grid(clue_grid), color="black", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{seed['hardest_strategy']}\nrating={seed.get('rating', 'n/a')}\nsolver_steps={seed.get('num_steps_solver', 'n/a')}", fontsize=8)
        if r == 0:
            ax.set_ylabel("clues", fontsize=9)
        # Column 1: solver fill-time.
        ax = axes[r, 1]
        if solver_fs is not None:
            im = ax.imshow(solver_fs, cmap=cmap)
            overlay_sudoku_digits_on_heatmap_ax(ax, to_label_grid(answer), fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title("solver", fontsize=9)
        # One column per model.
        for k, obj in enumerate(OBJECTIVE_ORDER):
            ax = axes[r, 2 + k]
            mrow = df_eval[(df_eval["puzzle_hash"] == ph) & (df_eval["objective"] == obj)
                      & (df_eval["embed_tying"] == seed["embed_tying"]) & (df_eval["tau"] == target_tau)]
            if mrow.empty:
                ax.set_axis_off(); continue
            mrow = mrow.iloc[0]
            fs = grid_from_flat(mrow["fill_step_flat"])
            final = digit_grid_from_flat(mrow["final_grid_flat"])
            if fs is not None:
                ax.imshow(fs, cmap=cmap)
            digit_text = to_label_grid(final)
            for rr in range(9):
                for cc in range(9):
                    correct = int(answer[rr, cc]) == int(final[rr, cc]) and int(final[rr, cc]) != 0
                    color = "white" if correct else "red"
                    if digit_text[rr, cc] != "":
                        ax.text(cc, rr, digit_text[rr, cc], ha="center", va="center",
                                color=color, fontsize=7,
                                path_effects=[plt.matplotlib.patheffects.withStroke(
                                    linewidth=1.4, foreground="black")])
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(OBJECTIVE_LABEL[obj], fontsize=9)
    fig.suptitle("F5 — Qualitative gallery (red digits = wrong final cell)", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGS / "F5_qualitative.png", dpi=180, bbox_inches="tight")
    plt.show()

render_gallery(pick_one_per_tier())

## Matched-NFE comparison ("more steps" vs "smarter steps")

For each puzzle and each model, interpolate the threshold sweep to a target rollout
budget B and compare exact match at the *same* NFE. Decouples *more steps* from
*better steps*.

**Caveat (and follow-up #3 from the prior interpretation):** with the default
3-point τ sweep `{0.05, 0.15, 0.25}` the per-(model, tying) NFE ranges barely overlap
— LEAR/Relay sit around NFE ≈ 7-9, MLM/No-Relay sit around NFE ≈ 13-19. So a single
B will only have bracketed interpolation for a subset of models. The sanity table
below shows, per (model, tying), the τ → NFE range so you can see where the τ
sweep actually overlaps. Re-dump with extra τs (e.g. `THRESHOLDS="0.01 0.05 0.15 0.25 0.35 0.45"`)
to fix this.

In [ ]:
def per_model_nfe_range(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Per-(objective, tying) τ → NFE range. Tells you where matched-NFE
    interpolation will actually be bracketed."""
    rows = []
    for (obj, tying), sub in eval_df.groupby(["objective", "embed_tying"], observed=True):
        per_tau = (
            sub.groupby("tau", observed=True)
            .agg(em=("exact_match", "mean"), nfe=("rollout_steps", "mean"))
            .reset_index()
            .sort_values("tau")
        )
        if per_tau.empty:
            continue
        rows.append({
            "objective": OBJECTIVE_LABEL.get(obj, obj),
            "embed_tying": tying,
            "tau_min": per_tau["tau"].min(),
            "tau_max": per_tau["tau"].max(),
            "nfe_at_tau_min": per_tau["nfe"].iloc[0],
            "nfe_at_tau_max": per_tau["nfe"].iloc[-1],
            "em_at_tau_min": per_tau["em"].iloc[0],
            "em_at_tau_max": per_tau["em"].iloc[-1],
            "n_taus": int(per_tau.shape[0]),
        })
    return pd.DataFrame(rows)

print("=== τ → NFE range diagnostic (per model × tying) ===")
nfe_diag = per_model_nfe_range(df_eval)
print(nfe_diag.round(2).to_string(index=False))

def matched_nfe(metric_col: str, B: float, eps: float = 1.5) -> pd.DataFrame:
    rows = []
    for (obj, tying), sub in df_eval.groupby(["objective", "embed_tying"], observed=True):
        for ph, g in sub.groupby("puzzle_hash"):
            if g[metric_col].isna().all():
                continue
            xs = g["rollout_steps"].astype(float).values
            ys = g[metric_col].astype(float).values
            if len(xs) == 0:
                continue
            order = np.argsort(xs)
            xs = xs[order]; ys = ys[order]
            bracketed = (xs.min() - eps) <= B <= (xs.max() + eps)
            if not bracketed:
                continue
            y = float(np.interp(B, xs, ys))
            rows.append({"objective": obj, "embed_tying": tying,
                          "puzzle_hash": ph, metric_col: y})
    return pd.DataFrame(rows)

# Pick the matched-NFE budgets from observed NFE percentiles instead of hardcoding,
# so this is robust to whatever τ values the dump actually has.
nfe_obs = (
    df_eval.groupby(["objective", "embed_tying", "tau"], observed=True)["rollout_steps"]
    .mean()
)
nfe_lo = float(nfe_obs.min())
nfe_hi = float(nfe_obs.max())
B_grid = np.linspace(nfe_lo, nfe_hi, 5).round(1).tolist()
print(f"\nB grid (auto-picked from observed NFE range [{nfe_lo:.1f}, {nfe_hi:.1f}]): {B_grid}")

n_total_puzzles = df_eval["puzzle_hash"].nunique()
for B in B_grid:
    em = matched_nfe("exact_match", B)
    if em.empty:
        print(f"\n=== Matched NFE = {B} === no model is bracketed at this NFE ===")
        continue
    summary = (
        em.groupby(["objective", "embed_tying"], observed=True)["exact_match"]
        .agg(["mean", "count"])
        .rename(columns={"mean": f"em@NFE={B}", "count": f"n@NFE={B}"})
    )
    coverage = summary[f"n@NFE={B}"] / n_total_puzzles
    print(f"\n=== Matched NFE = {B} (coverage out of {n_total_puzzles}) ===")
    print(summary.unstack("embed_tying"))
    print("coverage:")
    print(coverage.unstack("embed_tying").round(2))

## Optional — single-puzzle deep dive

Pick any `puzzle_hash` from `df` and inspect every model side-by-side, including
step-by-step deltas and the dominant primitive at the first commit step.

In [ ]:
if not df_eval.empty:
    PH = df_eval["puzzle_hash"].iloc[0]
    rows = df_eval[df_eval["puzzle_hash"] == PH]
    print("clue:", rows["question"].iloc[0])
    print("answer:", rows["truth_text"].iloc[0])
    print(f"hardest_strategy: {rows['hardest_strategy'].iloc[0]}",
          f"| n_clues: {rows['n_clues'].iloc[0]}",
          f"| rating: {rows['rating'].iloc[0]}")
    summary = rows[[
        "objective", "embed_tying", "tau",
        "exact_match", "rollout_steps",
        "violations_total_steps", "final_violations",
        "first_step_naked_single_rate",
        "first_step_hidden_single_rate",
        "first_step_advanced_rate",
        "fill_order_spearman", "prefix_jaccard_at_5",
    ]]
    print(summary.to_string(index=False))